In [0]:
df = spark.table("default.cleaned_rides_forecasting_ready")
df.show(5)


+---------+-------------------+------------+-----------+-----------+-----------+------------+--------+
|  ride_id|           datetime| pickup_area|  drop_area|fare_amount|ride_status|service_type|distance|
+---------+-------------------+------------+-----------+-----------+-----------+------------+--------+
|BKG000001|15-01-2024 14:15:00| Benson Town|Magadi Road|     881.71|    Success|       eBike|    2.52|
|BKG000002|28-01-2024 09:50:00|    Cox Town|Indiranagar|        0.0| Incomplete|        Moto|     0.0|
|BKG000003|06-01-2024 03:33:00|Mahadevapura| Whitefield|     1458.3|    Success|    Go Sedan|    26.2|
|BKG000004|07-01-2024 01:28:00|    RT Nagar| Nagarbhavi|    1323.55|    Success|    Go Sedan|    26.2|
|BKG000005|07-01-2024 07:32:00|Basavanagudi|Vijayanagar|    1999.27|    Success|     Uber Go|   27.58|
+---------+-------------------+------------+-----------+-----------+-----------+------------+--------+
only showing top 5 rows


In [0]:
from pyspark.sql import functions as F

# Load table
df = spark.table("default.cleaned_rides_forecasting_ready")

# Normalization function (first 2 words)
def normalize(col):
    return F.trim(
        F.concat_ws(" ",
            F.split(col, " ").getItem(0),
            F.split(col, " ").getItem(1)
        )
    )

# Create normalized columns
df = df.withColumn("pickup_area_norm", normalize(F.col("pickup_area"))) \
       .withColumn("drop_area_norm", normalize(F.col("drop_area")))

# Replace original columns with normalized ones
df = df.drop("pickup_area", "drop_area") \
       .withColumnRenamed("pickup_area_norm", "pickup_area") \
       .withColumnRenamed("drop_area_norm", "drop_area")

# Preview
df.show(20, truncate=False)


+---------+-------------------+-----------+---------------------+------------+--------+-----------------+-----------------+
|ride_id  |datetime           |fare_amount|ride_status          |service_type|distance|pickup_area      |drop_area        |
+---------+-------------------+-----------+---------------------+------------+--------+-----------------+-----------------+
|BKG000001|15-01-2024 14:15:00|881.71     |Success              |eBike       |2.52    |Benson Town      |Magadi Road      |
|BKG000002|28-01-2024 09:50:00|0.0        |Incomplete           |Moto        |0.0     |Cox Town         |Indiranagar      |
|BKG000003|06-01-2024 03:33:00|1458.3     |Success              |Go Sedan    |26.2    |Mahadevapura     |Whitefield       |
|BKG000004|07-01-2024 01:28:00|1323.55    |Success              |Go Sedan    |26.2    |RT Nagar         |Nagarbhavi       |
|BKG000005|07-01-2024 07:32:00|1999.27    |Success              |Uber Go     |27.58   |Basavanagudi     |Vijayanagar      |
|BKG0000

In [0]:
from pyspark.sql import functions as F

# 1. Load original table
loc = spark.table("default.location_full")

# 2. Normalization function (first 2 words)
def normalize(col):
    return F.trim(
        F.concat_ws(" ",
            F.split(col, " ").getItem(0),
            F.split(col, " ").getItem(1)
        )
    )

# 3. Add normalized column
loc_norm = loc.withColumn("area_norm", normalize(F.col("location")))

# 4. Write as a NEW TABLE
loc_norm.write.mode("overwrite").saveAsTable("default.location_normalized")

# 5. Preview
spark.table("default.location_normalized").show(30, truncate=False)


+-----------------+----------+----------+-----------------+
|location         |lat       |lon       |area_norm        |
+-----------------+----------+----------+-----------------+
|Electronic City  |12.8487599|77.648253 |Electronic City  |
|Banashankari     |12.9278196|77.556621 |Banashankari     |
|Benson Town      |12.997343 |77.6036815|Benson Town      |
|Ulsoor           |12.9778793|77.6246697|Ulsoor           |
|Vijayanagar      |12.9709537|77.5373851|Vijayanagar      |
|Kadugodi         |12.9957428|77.7579489|Kadugodi         |
|Bannerghatta Road|12.9502614|77.6033252|Bannerghatta Road|
|Malleswaram      |13.0027353|77.5703253|Malleswaram      |
|Kudlu Gate       |12.8899174|77.6396808|Kudlu Gate       |
|Cox Town         |12.9954832|77.6239259|Cox Town         |
|Hebbal           |13.0382184|77.5919   |Hebbal           |
|Bellandur        |12.9371445|77.6720227|Bellandur        |
|Magadi Road      |12.9756527|77.5553548|Magadi Road      |
|BTM Layout       |12.9140008|77.6102821

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# 1️⃣ Convert datetime string → proper timestamp
df = df.withColumn(
    "datetime",
    F.to_timestamp("datetime", "dd-MM-yyyy HH:mm:ss")
)

# 2️⃣ Time zone function (safe for None)
def hour_to_zone(h):
    if h is None:
        return None
    if 0 <= h <= 4:
        return "late_night"
    if 5 <= h <= 7:
        return "early_morning"
    if 8 <= h <= 10:
        return "morning"
    if 11 <= h <= 14:
        return "midday"
    if 15 <= h <= 17:
        return "afternoon"
    if 18 <= h <= 20:
        return "evening"
    if 21 <= h <= 23:
        return "night"
    return None

hour_to_zone_udf = F.udf(hour_to_zone, T.StringType())

# 3️⃣ Extract date components + time_zone
df = (
    df.withColumn("year", F.year("datetime"))
      .withColumn("month", F.month("datetime"))
      .withColumn("day", F.dayofmonth("datetime"))
      .withColumn("hour", F.hour("datetime"))
      .withColumn("date", F.to_date("datetime"))
      .withColumn("time_zone", hour_to_zone_udf(F.col("hour")))
)

# 4️⃣ Preview
df.select("datetime", "date", "hour", "time_zone") \
  .show(20, truncate=False)


+-------------------+----------+----+-------------+
|datetime           |date      |hour|time_zone    |
+-------------------+----------+----+-------------+
|2024-01-15 14:15:00|2024-01-15|14  |midday       |
|2024-01-28 09:50:00|2024-01-28|9   |morning      |
|2024-01-06 03:33:00|2024-01-06|3   |late_night   |
|2024-01-07 01:28:00|2024-01-07|1   |late_night   |
|2024-01-07 07:32:00|2024-01-07|7   |early_morning|
|2024-01-28 11:31:00|2024-01-28|11  |midday       |
|2024-01-20 04:00:00|2024-01-20|4   |late_night   |
|2024-01-27 21:17:00|2024-01-27|21  |night        |
|2024-01-18 20:49:00|2024-01-18|20  |evening      |
|2024-01-24 22:19:00|2024-01-24|22  |night        |
|2024-01-13 12:07:00|2024-01-13|12  |midday       |
|2024-01-29 22:30:00|2024-01-29|22  |night        |
|2024-01-28 02:30:00|2024-01-28|2   |late_night   |
|2024-01-20 17:48:00|2024-01-20|17  |afternoon    |
|2024-01-25 11:33:00|2024-01-25|11  |midday       |
|2024-01-24 09:16:00|2024-01-24|9   |morning      |
|2024-01-14 

In [0]:
df = df.drop("distance")

df.printSchema()
df.show(20, truncate=False)


root
 |-- ride_id: string (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- ride_status: string (nullable = true)
 |-- pickup_area: string (nullable = false)
 |-- drop_area: string (nullable = false)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- time_zone: string (nullable = true)

+---------+-------------------+---------------------+-----------------+-----------------+----+-----+---+----+----------+-------------+
|ride_id  |datetime           |ride_status          |pickup_area      |drop_area        |year|month|day|hour|date      |time_zone    |
+---------+-------------------+---------------------+-----------------+-----------------+----+-----+---+----+----------+-------------+
|BKG000001|2024-01-15 14:15:00|Success              |Benson Town      |Magadi Road      |2024|1    |15 |14  |2024-01-15|midday       |
|BKG000002|2024-01-2

In [0]:
from pyspark.sql import functions as F

# ---------- 1. DEMAND ----------
df_demand = (
    df.groupBy("date", "pickup_area", "time_zone")
      .agg(F.count("*").alias("demand"))
      .withColumnRenamed("pickup_area", "area")
)

# ---------- 2. AVAILABLE CABS ----------
df_available = (
    df.groupBy("date", "drop_area", "time_zone")
      .agg(F.count("*").alias("available_cabs"))
      .withColumnRenamed("drop_area", "area")
)

# ---------- 3. MERGE BOTH ----------
df_zone_stats = (
    df_demand.join(df_available, on=["date", "area", "time_zone"], how="left")
             .fillna({"available_cabs": 0})
)

df_zone_stats = df_zone_stats.withColumn(
    "shortage", F.col("demand") - F.col("available_cabs")
)

# ---------- 4. JOIN STATS BACK TO ORIGINAL RIDE-LEVEL DF ----------
df_final = (
    df.join(
        df_zone_stats,
        (df["date"] == df_zone_stats["date"]) &
        (df["time_zone"] == df_zone_stats["time_zone"]) &
        (df["pickup_area"] == df_zone_stats["area"]),
        "left"
    )
)

# ---------- 5. CLEAN DUPLICATE COLUMNS ----------
df_final = df_final.drop(df_zone_stats["date"]) \
                   .drop(df_zone_stats["time_zone"]) \
                   .drop(df_zone_stats["area"])

# ---------- 6. SHOW FINAL OUTPUT ----------
df_final.show(50, truncate=False)
df_final.printSchema()


+---------+-------------------+---------------------+-----------------+-----------------+----+-----+---+----+----------+-------------+------+--------------+--------+
|ride_id  |datetime           |ride_status          |pickup_area      |drop_area        |year|month|day|hour|date      |time_zone    |demand|available_cabs|shortage|
+---------+-------------------+---------------------+-----------------+-----------------+----+-----+---+----+----------+-------------+------+--------------+--------+
|BKG000045|2024-01-01 12:37:00|Success              |Bellandur        |Hulimavu         |2024|1    |1  |12  |2024-01-01|midday       |25    |17            |8       |
|BKG000041|2024-01-01 02:19:00|Success              |HSR Layout       |Kadugodi         |2024|1    |1  |2   |2024-01-01|late_night   |27    |21            |6       |
|BKG000050|2024-01-02 04:38:00|Incomplete           |KR Puram         |MG Road          |2024|1    |2  |4   |2024-01-02|late_night   |29    |37            |-8      |
|BKG

In [0]:
df_final.write.mode("overwrite").saveAsTable("default.rides_enriched_demand_supply_version1")


In [0]:
df_export = spark.table("default.rides_enriched_demand_supply_version1")


In [0]:
output_path = "dbfs:/FileStore/rides_enriched_demand_supply_version1"

df_export.coalesce(1).write.mode("overwrite").csv(output_path, header=True)


In [0]:
files = dbutils.fs.ls("dbfs:/FileStore/rides_enriched_demand_supply_version1")
files


[FileInfo(path='dbfs:/FileStore/rides_enriched_demand_supply_version1/_SUCCESS', name='_SUCCESS', size=0, modificationTime=1763403975000),
 FileInfo(path='dbfs:/FileStore/rides_enriched_demand_supply_version1/_committed_3955878289500482249', name='_committed_3955878289500482249', size=113, modificationTime=1763403975000),
 FileInfo(path='dbfs:/FileStore/rides_enriched_demand_supply_version1/_started_3955878289500482249', name='_started_3955878289500482249', size=0, modificationTime=1763403972000),
 FileInfo(path='dbfs:/FileStore/rides_enriched_demand_supply_version1/part-00000-tid-3955878289500482249-e24d5158-ca16-4da3-aa0a-be8cd0391d2d-199-1-c000.csv', name='part-00000-tid-3955878289500482249-e24d5158-ca16-4da3-aa0a-be8cd0391d2d-199-1-c000.csv', size=39397111, modificationTime=1763403975000)]

In [0]:
csv_file = "dbfs:/FileStore/rides_enriched_demand_supply_version1/part-00000-tid-3955878289500482249-e24d5158-ca16-4da3-aa0a-be8cd0391d2d-199-1-c000.csv"

download_url = csv_file.replace("dbfs:/FileStore", "/files")
print(download_url)


/files/rides_enriched_demand_supply_version1/part-00000-tid-3955878289500482249-e24d5158-ca16-4da3-aa0a-be8cd0391d2d-199-1-c000.csv


In [0]:
loc_export = spark.table("default.location_normalized")
loc_export.show(5)


+---------------+----------+----------+---------------+
|       location|       lat|       lon|      area_norm|
+---------------+----------+----------+---------------+
|Electronic City|12.8487599| 77.648253|Electronic City|
|   Banashankari|12.9278196| 77.556621|   Banashankari|
|    Benson Town| 12.997343|77.6036815|    Benson Town|
|         Ulsoor|12.9778793|77.6246697|         Ulsoor|
|    Vijayanagar|12.9709537|77.5373851|    Vijayanagar|
+---------------+----------+----------+---------------+
only showing top 5 rows


In [0]:
output_path = "dbfs:/FileStore/location_normalized_export"

loc_export.coalesce(1).write.mode("overwrite").csv(output_path, header=True)


In [0]:
files = dbutils.fs.ls("dbfs:/FileStore/location_normalized_export")
files


[FileInfo(path='dbfs:/FileStore/location_normalized_export/_SUCCESS', name='_SUCCESS', size=0, modificationTime=1763404364000),
 FileInfo(path='dbfs:/FileStore/location_normalized_export/_committed_49310599203009221', name='_committed_49310599203009221', size=111, modificationTime=1763404364000),
 FileInfo(path='dbfs:/FileStore/location_normalized_export/_started_49310599203009221', name='_started_49310599203009221', size=0, modificationTime=1763404364000),
 FileInfo(path='dbfs:/FileStore/location_normalized_export/part-00000-tid-49310599203009221-66602250-ad4f-4c62-95dc-ea0ea25bcfbe-201-1-c000.csv', name='part-00000-tid-49310599203009221-66602250-ad4f-4c62-95dc-ea0ea25bcfbe-201-1-c000.csv', size=14362, modificationTime=1763404364000)]